In [ ]:

import pandas as pd

# 加载训练集和测试集
train_file = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/monster/train.csv'
test_file = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/monster/test.csv'

train_df = pd.read_csv(train_file)
test_df = pd.read_csv(test_file)

# 查看训练集和测试集的基本信息
print("训练集信息：")
print(train_df.info())
print("\n训练集前几行：")
print(train_df.head())

print("\n测试集信息：")
print(test_df.info())
print("\n测试集前几行：")
print(test_df.head())


训练集信息：
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 296 entries, 0 to 295
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             296 non-null    int64  
 1   bone_length    296 non-null    float64
 2   rotting_flesh  296 non-null    float64
 3   hair_length    296 non-null    float64
 4   color          296 non-null    object 
 5   type           296 non-null    object 
dtypes: float64(3), int64(1), object(2)
memory usage: 14.0+ KB
None

训练集前几行：
    id  bone_length  rotting_flesh  hair_length  color    type
0  472     0.681615       0.529227     0.625242  white   Ghoul
1  170     0.480836       0.407930     0.539005  clear  Goblin
2  189     0.375197       0.742953     0.320764   blue   Ghost
3  861     0.626017       0.172182     0.408422   blue   Ghoul
4   30     0.250770       0.246258     0.554654  black   Ghost

测试集信息：
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75 entries, 0 to 74
Data

In [ ]:

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

# 特征编码
label_encoder_color = LabelEncoder()
label_encoder_type = LabelEncoder()

train_df['color_encoded'] = label_encoder_color.fit_transform(train_df['color'])
train_df['type_encoded'] = label_encoder_type.fit_transform(train_df['type'])

test_df['color_encoded'] = label_encoder_color.transform(test_df['color'])

# 选择特征和目标变量
X_train = train_df[['bone_length', 'rotting_flesh', 'hair_length', 'color_encoded']]
y_train = train_df['type_encoded']

X_test = test_df[['bone_length', 'rotting_flesh', 'hair_length', 'color_encoded']]

# 数据标准化
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 分割训练集和验证集
X_train_final, X_val, y_train_final, y_val = train_test_split(X_train_scaled, y_train, test_size=0.2, random_state=42)

# 查看处理后的数据
print("处理后的训练集特征：")
print(X_train_final[:5])
print("\n处理后的训练集目标变量：")
print(y_train_final[:5])
print("\n处理后的验证集特征：")
print(X_val[:5])
print("\n处理后的验证集目标变量：")
print(y_val[:5])


处理后的训练集特征：
[[-0.01113367 -0.49065011 -1.65974045 -0.25309767]
 [-0.20000694 -0.32158688  0.07409637  0.96506348]
 [-0.8587482  -1.98998706  0.38969561  0.96506348]
 [-0.689559   -1.66409312  0.33346911 -0.25309767]
 [ 0.67072614  0.63354002  2.44323189  0.96506348]]

处理后的训练集目标变量：
63     2
17     2
215    2
219    2
183    1
Name: type_encoded, dtype: int64

处理后的验证集特征：
[[-1.4218105  -0.68509099  0.07697681  0.3559829 ]
 [-1.70756803 -0.75169673 -0.60127801 -2.0803394 ]
 [-0.68373413 -0.76551954 -0.26967328  0.96506348]
 [-0.08538435 -1.73926051  1.88966928 -0.25309767]
 [ 1.05434126 -0.53703326  0.67245312 -2.0803394 ]]

处理后的验证集目标变量：
274    0
155    0
84     1
82     1
261    1
Name: type_encoded, dtype: int64


In [ ]:


from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 定义模型
models = {
    'Logistic Regression': LogisticRegression(),
    'Decision Tree': DecisionTreeClassifier(),
    'Random Forest': RandomForestClassifier()
}

# 训练和评估模型
best_model = None
best_accuracy = 0

for name, model in models.items():
    model.fit(X_train_final, y_train_final)
    y_val_pred = model.predict(X_val)
    val_accuracy = accuracy_score(y_val, y_val_pred)
    print(f"{name} - 验证集准确率: {val_accuracy:.4f}")
    if val_accuracy > best_accuracy:
        best_accuracy = val_accuracy
        best_model = model

print(f"最佳模型: {best_model.__class__.__name__}")


Logistic Regression - 验证集准确率: 0.6667
Decision Tree - 验证集准确率: 0.5667
Random Forest - 验证集准确率: 0.6167
最佳模型: LogisticRegression


In [ ]:


# 使用最佳模型进行测试集预测
y_test_pred = best_model.predict(X_test_scaled)

# 计算测试集上的准确率
y_test = label_encoder_type.transform(test_df['type'])
test_accuracy = accuracy_score(y_test, y_test_pred)
print(f"测试集准确率: {test_accuracy:.4f}")

# 将预测结果保存到文件中
test_df['predicted_type'] = label_encoder_type.inverse_transform(y_test_pred)
test_df[['id', 'predicted_type']].to_csv('monster_predictions.csv', index=False)

print("预测结果已保存到文件 monster_predictions.csv 中。")


测试集准确率: 0.5867
预测结果已保存到文件 monster_predictions.csv 中。
